# 可选实验 - 神经元与层
在本实验中，我们将探索神经元/单元和层的内部工作机制。特别是，本实验会把它与你在课程 1 中已掌握的模型——回归/线性模型和逻辑模型——做类比。本实验将介绍 Tensorflow，并演示这些模型在该框架中是如何实现的。
<figure>
   <img src="./图片/C2_W1_NeuronsAndLayers.png"  style="width:440px;height:200px;" >

> **图中文字（中文）**：左：生物神经元；右：神经元的简化数学模型 标注：输入（inputs）、细胞核（nucleus）、输出（outputs） 图片来源：https://biologydictionary.net/sensory-neuron/
</figure>


## 程序包
**Tensorflow 与 Keras**  
Tensorflow 是 Google 开发的机器学习程序包。2019 年，Google 把 Keras 集成进 Tensorflow，并发布了 Tensorflow 2.0。Keras 是 François Chollet 独立开发的框架，它为 Tensorflow 提供了一个简单的、以层为中心的接口。本课程将使用 Keras 接口。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras import Sequential
from tensorflow.keras.losses import MeanSquaredError, BinaryCrossentropy
from tensorflow.keras.activations import sigmoid
from lab_utils_common import dlc
from lab_neurons_utils import plt_prob_1d, sigmoidnp, plt_linear, plt_logistic
plt.style.use('./deeplearning.mplstyle')
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

## 无激活函数的神经元 - 回归/线性模型

### 数据集
我们将使用课程 1 中的一个例子：房价的线性回归。

In [ ]:
X_train = np.array([[1.0], [2.0]], dtype=np.float32)           #（面积，单位：千平方英尺）
Y_train = np.array([[300.0], [500.0]], dtype=np.float32)       #（价格，单位：千美元）

fig, ax = plt.subplots(1,1)
ax.scatter(X_train, Y_train, marker='x', c='r', label="Data Points")
ax.legend( fontsize='xx-large')
ax.set_ylabel('Price (in 1000s of dollars)', fontsize='xx-large')
ax.set_xlabel('Size (1000 sqft)', fontsize='xx-large')
plt.show()

### 回归/线性模型
无激活函数的神经元所实现的函数与课程 1 中的线性回归相同：
$$ f_{\\mathbf{w},b}(x^{(i)}) = \\mathbf{w}\\cdot x^{(i)} + b \\tag{1}$$



我们可以定义一个只含一个神经元（单元）的层，并把它与我们熟悉的线性回归函数做比较。

In [ ]:
linear_layer = tf.keras.layers.Dense(units=1, activation = 'linear', )

让我们检查一下权重。

In [ ]:
linear_layer.get_weights()

此时还没有权重，因为权重尚未被实例化。让我们在 `X_train` 中的一个样本上试用这个模型。这会触发权重的实例化。注意，层的输入必须是二维的，所以我们要对它做 reshape。

In [ ]:
a1 = linear_layer(X_train[0].reshape(1,1))
print(a1)

结果是一个形状为 (1,1) 的张量（数组的另一种叫法），即只有一个条目。  
现在我们看看权重和偏置。这些权重被随机初始化为较小的数，偏置默认初始化为零。

In [ ]:
w, b= linear_layer.get_weights()
print(f"w = {w}, b={b}")

只有单个输入特征的线性回归模型 (1) 会有一个权重和一个偏置。这与上面 `linear_layer` 的维度一致。  

权重被初始化为随机值，所以让我们把它们设为一些已知值。

In [ ]:
set_w = np.array([[200]])
set_b = np.array([100])

# set_weights 接收一个 numpy 数组列表
linear_layer.set_weights([set_w, set_b])
print(linear_layer.get_weights())

让我们把公式 (1) 与层的输出做比较。

In [ ]:
a1 = linear_layer(X_train[0].reshape(1,1))
print(a1)
alin = np.dot(set_w,X_train[0].reshape(1,1)) + set_b
print(alin)

它们产生的结果完全相同！
现在，我们可以用这个线性层在训练数据上做预测了。

In [ ]:
prediction_tf = linear_layer(X_train)
prediction_np = np.dot( X_train, set_w) + set_b

In [ ]:
plt_linear(X_train, Y_train, prediction_tf, prediction_np)

## 带 Sigmoid 激活函数的神经元
带 sigmoid 激活函数的神经元/单元所实现的函数与课程 1 中的逻辑回归相同：
$$ f_{\\mathbf{w},b}(x^{(i)}) = g(\\mathbf{w}x^{(i)} + b) \\tag{2}$$
其中 $$g(x) = sigmoid(x)$$ 

让我们把 $w$ 和 $b$ 设为一些已知值，并检查这个模型。


### 数据集
我们将使用课程 1 中的一个例子：逻辑回归。

In [ ]:
X_train = np.array([0., 1, 2, 3, 4, 5], dtype=np.float32).reshape(-1,1)  # 二维矩阵
Y_train = np.array([0,  0, 0, 1, 1, 1], dtype=np.float32).reshape(-1,1)  # 二维矩阵

In [ ]:
pos = Y_train == 1
neg = Y_train == 0
X_train[pos]

In [ ]:
pos = Y_train == 1
neg = Y_train == 0

fig,ax = plt.subplots(1,1,figsize=(4,3))
ax.scatter(X_train[pos], Y_train[pos], marker='x', s=80, c = 'red', label="y=1")
ax.scatter(X_train[neg], Y_train[neg], marker='o', s=100, label="y=0", facecolors='none', 
              edgecolors=dlc["dlblue"],lw=3)

ax.set_ylim(-0.08,1.1)
ax.set_ylabel('y', fontsize=12)
ax.set_xlabel('x', fontsize=12)
ax.set_title('one variable plot')
ax.legend(fontsize=12)
plt.show()

### 逻辑神经元
我们可以通过加上 sigmoid 激活函数来实现一个“逻辑神经元”。此时神经元的功能由上面的 (2) 描述。  
本节将创建一个包含逻辑层的 Tensorflow 模型，以演示另一种创建模型的方法。Tensorflow 最常用于创建多层模型。[Sequential](https://keras.io/guides/sequential_model/) 模型是构建这类模型的一种便捷方式。

In [ ]:
model = Sequential(
    [
        tf.keras.layers.Dense(1, input_dim=1,  activation = 'sigmoid', name='L1')
    ]
)

`model.summary()` 显示模型中的层和参数数量。这个模型只有一层，且该层只有一个单元。该单元有两个参数：$w$ 和 $b$。

In [ ]:
model.summary()

In [ ]:
logistic_layer = model.get_layer('L1')
w,b = logistic_layer.get_weights()
print(w,b)
print(w.shape,b.shape)

让我们把权重和偏置设为一些已知值。

In [ ]:
set_w = np.array([[2]])
set_b = np.array([-4.5])
# set_weights 接收一个 numpy 数组列表
logistic_layer.set_weights([set_w, set_b])
print(logistic_layer.get_weights())

让我们把公式 (2) 与层的输出做比较。

In [ ]:
a1 = model.predict(X_train[0].reshape(1,1))
print(a1)
alog = sigmoidnp(np.dot(set_w,X_train[0].reshape(1,1)) + set_b)
print(alog)

它们产生的结果完全相同！
现在，我们可以用这个逻辑层和 NumPy 模型在训练数据上做预测了。

In [ ]:
plt_logistic(X_train, Y_train, model, set_w, set_b, pos, neg)

上面的阴影反映了 sigmoid 的输出，它在 0 到 1 之间变化。

# 恭喜！
你构建了一个非常简单的神经网络，并探索了神经元与课程 1 中线性回归、逻辑回归的相似之处。